In [1]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


# =========================
# 1. Stieltjes–Wigert polynomials (monic, KLS 3.27)
# =========================

def stieltjes_wigert_polynomials(x, degree, q_param):
    """
    Exact Stieltjes–Wigert monic polynomials P_n(x) via the standard
    three-term recurrence (Koekoek–Lesky–Swarttouw, section 3.27):

        x P_n(x) = P_{n+1}(x) + b_n P_n(x) + a_n^2 P_{n-1}(x),

    with
        b_n  = q^(-2n - 3/2) * (1 + q - q^{n+1}),
        a_n^2 = q^(-4n) * (1 - q^n),

    and initial conditions
        P_{-1}(x) = 0,  P_0(x) = 1.
    """
    # q in (0.005, 0.995) for numerical stability
    q = torch.sigmoid(q_param) * 0.99 + 0.005

    # Map raw x to (0, +inf); SW weight is on (0, +inf)
    x_pos = torch.exp(2.0 * torch.tanh(x))  # (e^-2, e^2) ⊂ (0, ∞)

    # P_{-1}=0, P_0=1
    Pm1 = torch.zeros_like(x_pos)
    P0 = torch.ones_like(x_pos)

    polys = [P0]

    if degree >= 1:
        # n=0 -> b_0 = q^(-3/2) * (1 + q - q^1) = q^(-3/2)
        b0 = q ** (-1.5)
        P1 = x_pos - b0
        polys.append(P1)
        Pm1, P0 = P0, P1

    for n in range(1, degree):
        # b_n, a_n^2 for SW monic
        bn = q ** (-2.0 * n - 1.5) * (1.0 + q - q ** (n + 1))
        an2 = q ** (-4.0 * n) * (1.0 - q ** n)

        # P_{n+1} = x P_n - b_n P_n - a_n^2 P_{n-1}
        P1 = x_pos * P0 - bn * P0 - an2 * Pm1

        polys.append(P1)
        Pm1, P0 = P0, P1

    return torch.stack(polys, dim=-1)


class StieltjesWigertKANLayer(nn.Module):
    def __init__(self, input_dim, output_dim, degree=3, learnable_q=True):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.degree = degree

        # [output_dim, input_dim, degree+1]
        self.weights = nn.Parameter(
            torch.randn(output_dim, input_dim, degree + 1) * 0.02
        )

        if learnable_q:
            self.q_param = nn.Parameter(torch.tensor(0.0))
        else:
            self.register_buffer('q_param', torch.tensor(0.0))

    def forward(self, x):
        if x.dim() > 2:
            x = x.view(x.shape[0], -1)

        polys = stieltjes_wigert_polynomials(x, self.degree, self.q_param)
        # b i d, o i d -> b o
        out = torch.einsum('bid, oid -> bo', polys, self.weights)
        return out


class StieltjesWigertKAN_MNIST(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=64, num_classes=10, degree=3):
        super().__init__()
        self.flatten = nn.Flatten()

        self.kan1 = StieltjesWigertKANLayer(input_dim, hidden_dim, degree=degree)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.drop1 = nn.Dropout(0.1)

        self.kan2 = StieltjesWigertKANLayer(hidden_dim, hidden_dim, degree=degree)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.drop2 = nn.Dropout(0.1)

        self.kan3 = StieltjesWigertKANLayer(hidden_dim, num_classes, degree=degree)

    def forward(self, x):
        x = self.flatten(x)
        x = self.kan1(x)
        x = self.norm1(x)
        x = self.drop1(x)

        x = self.kan2(x)
        x = self.norm2(x)
        x = self.drop2(x)

        x = self.kan3(x)
        return x


# =========================
# 2. Data loaders (use precomputed MNIST stats)
# =========================

MNIST_MEAN = (0.1307,)  # استاندارد برای MNIST[web:21][web:19]
MNIST_STD = (0.3081,)


def make_mnist_loaders(batch_size=64, device='cpu'):
    is_cuda = (device != 'cpu')

    train_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MNIST_MEAN, MNIST_STD)
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MNIST_MEAN, MNIST_STD)
    ])

    train_ds = datasets.MNIST('./data_mnist',
                              train=True,
                              download=True,
                              transform=train_tf)
    test_ds = datasets.MNIST('./data_mnist',
                             train=False,
                             download=True,
                             transform=test_tf)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,          # >0 برای سرعت I/O[web:76][web:111]
        pin_memory=is_cuda      # برای GPU مفید است[web:104][web:106][web:107][web:113]
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=1000,
        shuffle=False,
        num_workers=4,
        pin_memory=is_cuda
    )
    return train_loader, test_loader


# =========================
# 3. Training loop for MNIST
# =========================

def train_mnist(epochs=10, batch_size=32, device='cuda'):
    print("=" * 60)
    print("Stieltjes–Wigert KAN on MNIST (precomputed norm, fast I/O)")
    print(f"Device: {device} | Epochs: {epochs} | Batch: {batch_size}")
    print("=" * 60)

    train_loader, test_loader = make_mnist_loaders(
        batch_size=batch_size, device=device
    )

    model = StieltjesWigertKAN_MNIST(
        input_dim=784, hidden_dim=32, num_classes=10, degree=3
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    best_test_acc = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        start = time.time()
        epoch_loss = 0.0

        for data, target in train_loader:
            data = data.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        # eval on test
        model.eval()
        correct_te, total_te = 0, 0
        with torch.no_grad():
            for data, target in test_loader:
                data = data.to(device, non_blocking=True)
                target = target.to(device, non_blocking=True)
                pred = model(data).argmax(dim=1)
                correct_te += pred.eq(target).sum().item()
                total_te += target.size(0)
        test_acc = 100.0 * correct_te / total_te
        best_test_acc = max(best_test_acc, test_acc)

        avg_loss = epoch_loss / len(train_loader)
        elapsed = time.time() - start

        # log q values
        q_info = []
        for name, p in model.named_parameters():
            if 'q_param' in name:
                q_val = (torch.sigmoid(p) * 0.99 + 0.005).item()
                q_info.append(f"{name}: q={q_val:.4f}")

        print(f"Epoch {epoch:2d}/{epochs} | Loss: {avg_loss:.4f} | "
              f"Test Acc: {test_acc:.2f}% | Best Test: {best_test_acc:.2f}% "
              f"| Time: {elapsed:.1f}s")
        if q_info:
            print(" ", " | ".join(q_info))

        scheduler.step()

    return model


# =========================
# 4. Main
# =========================

if __name__ == '__main__':
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    EPOCHS = 50
    BATCH_SIZE = 64

    model = train_mnist(epochs=EPOCHS,
                        batch_size=BATCH_SIZE,
                        device=DEVICE)

    torch.save(model.state_dict(), 'sw_kan_mnist_exact_fastio.pth')
    print("Model saved: sw_kan_mnist_exact_fastio.pth")


Stieltjes–Wigert KAN on MNIST (precomputed norm, fast I/O)
Device: cuda | Epochs: 50 | Batch: 64


100%|██████████| 9.91M/9.91M [00:00<00:00, 16.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 485kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.41MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.55MB/s]


Epoch  1/50 | Loss: 2.9380 | Test Acc: 92.15% | Best Test: 92.15% | Time: 13.6s
  kan1.q_param: q=0.4600 | kan2.q_param: q=0.5256 | kan3.q_param: q=0.6042
Epoch  2/50 | Loss: 0.3223 | Test Acc: 94.48% | Best Test: 94.48% | Time: 12.5s
  kan1.q_param: q=0.4568 | kan2.q_param: q=0.5625 | kan3.q_param: q=0.6800
Epoch  3/50 | Loss: 0.2278 | Test Acc: 94.90% | Best Test: 94.90% | Time: 12.6s
  kan1.q_param: q=0.4535 | kan2.q_param: q=0.5745 | kan3.q_param: q=0.7058
Epoch  4/50 | Loss: 0.1889 | Test Acc: 95.83% | Best Test: 95.83% | Time: 12.7s
  kan1.q_param: q=0.4538 | kan2.q_param: q=0.5819 | kan3.q_param: q=0.7180
Epoch  5/50 | Loss: 0.1681 | Test Acc: 96.40% | Best Test: 96.40% | Time: 12.7s
  kan1.q_param: q=0.4527 | kan2.q_param: q=0.5854 | kan3.q_param: q=0.7257
Epoch  6/50 | Loss: 0.1550 | Test Acc: 96.34% | Best Test: 96.40% | Time: 12.8s
  kan1.q_param: q=0.4564 | kan2.q_param: q=0.5870 | kan3.q_param: q=0.7314
Epoch  7/50 | Loss: 0.1451 | Test Acc: 96.71% | Best Test: 96.71% | Ti